## 1. 가장 큰 상처 순으로 파일 정리하기

In [1]:
import os
import cv2
import numpy as np
import pandas as pd

ROOT = "data_wound_seg"
MASK_DIR = os.path.join(ROOT, "train_masks")
IMG_DIR  = os.path.join(ROOT, "train_images")
OUT_DIR  = os.path.join(ROOT, "out")
os.makedirs(OUT_DIR, exist_ok=True)

rows = []
for fn in os.listdir(MASK_DIR):
    if not fn.lower().endswith(".png"):
        continue
    mask_path = os.path.join(MASK_DIR, fn)
    m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if m is None:
        continue
    area = int((m > 127).sum())  # 상처 면적(픽셀)
    rows.append((fn, area))

df = pd.DataFrame(rows, columns=["filename", "area"]).sort_values("area", ascending=False)
df.to_csv(os.path.join(OUT_DIR, "mask_area_rank_train.csv"), index=False)

print("총 마스크 수:", len(df))
print("가장 큰 상처 TOP 1:")
print(df.head(1))
print("\n가장 큰 상처 TOP 20 저장됨 -> out/mask_area_rank_train.csv")


총 마스크 수: 2197
가장 큰 상처 TOP 1:
           filename    area
594  wsnet_1097.png  126188

가장 큰 상처 TOP 20 저장됨 -> out/mask_area_rank_train.csv


In [2]:
#test에서도 한 번 검증하기

import os
import cv2
import numpy as np
import pandas as pd

ROOT = "data_wound_seg"
MASK_DIR = os.path.join(ROOT, "test_masks")
IMG_DIR  = os.path.join(ROOT, "test_images")
OUT_DIR  = os.path.join(ROOT, "out")
os.makedirs(OUT_DIR, exist_ok=True)

rows = []
for fn in os.listdir(MASK_DIR):
    if not fn.lower().endswith(".png"):
        continue
    mask_path = os.path.join(MASK_DIR, fn)
    m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if m is None:
        continue
    area = int((m > 127).sum())  # 상처 면적(픽셀)
    rows.append((fn, area))

df = pd.DataFrame(rows, columns=["filename", "area"]).sort_values("area", ascending=False)
df.to_csv(os.path.join(OUT_DIR, "mask_area_rank_test.csv"), index=False)

print("총 마스크 수:", len(df))
print("가장 큰 상처 TOP 1:")
print(df.head(1))
print("\n가장 큰 상처 TOP 20 저장됨 -> out/mask_area_rank_test.csv")


총 마스크 수: 539
가장 큰 상처 TOP 1:
        filename   area
9  fusc_0995.png  34678

가장 큰 상처 TOP 20 저장됨 -> out/mask_area_rank_test.csv


## 2. pseudo-time series 구성

In [3]:
import os, math, random, csv
from dataclasses import dataclass
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
import cv2

# 경로 설정 
ROOT = "data_wound_seg"
IMG_DIR  = os.path.join(ROOT, "train_images")
MASK_DIR = os.path.join(ROOT, "train_masks")

RANK_CSV = os.path.join(ROOT, "out", "mask_area_rank_train.csv")  
OUT_META = os.path.join(ROOT, "out", "meta_train_cases.csv")


# 생성 옵션
SEED = 42
ANCHOR_TOP_Q = 0.15          # 상위 15%를 anchor 후보 풀로
NUM_CASES_NORMAL  = 50
NUM_CASES_PLATEAU = 15
NUM_CASES_REBOUND = 15

DAYS = [0, 3, 6, 9, 12, 15, 18]

RATIOS_NORMAL  = [1.00, 0.75, 0.55, 0.40, 0.28, 0.18, 0.10]
RATIOS_PLATEAU = [1.00, 0.78, 0.60, 0.45, 0.44, 0.44, 0.40]
RATIOS_REBOUND = [1.00, 0.78, 0.60, 0.48, 0.52, 0.35, 0.22]


# 데이터 구조
@dataclass
class Sample:
    filename: str
    img_path: str
    mask_path: str
    area: float #상처의 크기 (픽셀 수)
    circ: float #원형도
    ar: float #종횡비

# 마스크 (이미지 -> 이진)
def read_mask(mask_path: str) -> np.ndarray:
    m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if m is None:
        raise FileNotFoundError(mask_path)
    return (m > 127).astype(np.uint8)

#핵심 피처 계산
def compute_features(mask: np.ndarray) -> Tuple[float, float, float]:
    area = float(mask.sum())
    if area <= 0:
        return 0.0, 0.0, 0.0

    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return 0.0, 0.0, 0.0

    c = max(cnts, key=cv2.contourArea)
    peri = float(cv2.arcLength(c, True))
    circ = float(4.0 * math.pi * area / (peri * peri)) if peri > 1e-6 else 0.0

    x, y, w, h = cv2.boundingRect(c)
    ar = float(w / h) if h > 0 else 0.0
    return area, circ, ar

#sample 하나를 3차원 피처 벡터로 임베딩
#서로 다른 상처간 형태적 거리를 계산
def feat_vec(s: Sample) -> np.ndarray:
    return np.array([
        math.log(max(s.area, 1.0)),
        s.circ,
        math.log(max(s.ar, 1e-3))
    ], dtype=np.float32)

#두 sample을 feature space에서 비교해서 형태 거리 계산
#유클리드 거리
def dist(a: Sample, b: Sample) -> float:
    return float(np.linalg.norm(feat_vec(a) - feat_vec(b)))

#sample 객체 리스트 만듦
def load_samples_from_rank(rank_csv: str) -> List[Sample]:
    """
    사전에 생성된 마스크 면적 랭킹 CSV를 이용해 학습용 상처 샘플을 로드

    rank_csv는 다음 컬럼을 반드시 포함해야 함
    - filename: train_images 및 train_masks에 공통으로 존재하는 파일명
    - area: 마스크의 픽셀 면적 (정렬 및 검증 목적)

    유효하지 않은 파일 쌍이나 빈 마스크는 자동으로 제외
    """
    df = pd.read_csv(rank_csv)
    if "filename" not in df.columns:
        raise ValueError("rank_csv에 'filename' 컬럼이 필요")
    if "area" not in df.columns:
        raise ValueError("rank_csv에 'area' 컬럼이 필요")

    samples: List[Sample] = []
    missing = 0

    for fn in df["filename"].tolist():
        img_path = os.path.join(IMG_DIR, fn)
        mask_path = os.path.join(MASK_DIR, fn)

        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            missing += 1
            continue

        mask = read_mask(mask_path)
        area, circ, ar = compute_features(mask)
        if area <= 0:
            continue

        samples.append(Sample(fn, img_path, mask_path, area, circ, ar))

    print(f"[load_samples] loaded={len(samples)} missing_pairs={missing}")
    return samples


#목표 면적을 맞추면서 초기 상처와 형태적으로 가장 비슷한 샘플을 고름
def pick_best(candidates: List[Sample], anchor: Sample, target_area: float,
              used: set, alpha: float = 1.0, beta: float = 0.7) -> Optional[Sample]:
    best, best_score = None, 1e18
    log_t = math.log(max(target_area, 1.0))

    for s in candidates:
        if s.filename in used:
            continue
        area_term = abs(math.log(max(s.area, 1.0)) - log_t)
        shape_term = dist(s, anchor)
        score = alpha * area_term + beta * shape_term
        if score < best_score:
            best_score = score
            best = s

    return best

#한 환자의 시간 경과 케이스(시계열)를 실제로 만들어냄
def build_cases(samples_sorted_by_area_desc: List[Sample],
                ratios: List[float], days: List[int],
                num_cases: int, anchor_top_q: float) -> List[List[Tuple[Sample, int]]]:

    assert len(ratios) == len(days)
    samples = samples_sorted_by_area_desc

    n_anchor_pool = max(1, int(len(samples) * anchor_top_q))
    anchor_pool = samples[:n_anchor_pool]

    all_cases = []
    for _ in range(num_cases):
        anchor = random.choice(anchor_pool)
        used = {anchor.filename}
        case = [(anchor, days[0])]
        A0 = anchor.area

        ok = True
        for r, d in zip(ratios[1:], days[1:]):
            target = r * A0
            nxt = pick_best(samples, anchor, target_area=target, used=used)
            if nxt is None:
                ok = False
                break
            case.append((nxt, d))
            used.add(nxt.filename)

        if ok:
            all_cases.append(case)

    return all_cases

#시계열 케이스들을 메타데이터 파일로 정리
def save_meta(cases, out_csv: str):
    os.makedirs(os.path.dirname(out_csv), exist_ok=True)
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["case_id", "case_type", "day", "filename", "img_path", "mask_path", "area"])

        for ci, (case_type, case) in enumerate(cases):
            case_id = f"{ci:04d}"
            for s, day in case:
                w.writerow([case_id, case_type, day, s.filename, s.img_path, s.mask_path, int(s.area)])

    print(f"[save_meta] saved -> {out_csv}")

if __name__ == "__main__":
    #랜덤 시드 고정
    random.seed(SEED)

    #샘플 로드 / 정렬
    samples = load_samples_from_rank(RANK_CSV)
    samples = sorted(samples, key=lambda s: s.area, reverse=True)

    #케이스 생성
    cases_normal  = [("normal",  c) for c in build_cases(samples, RATIOS_NORMAL,  DAYS, NUM_CASES_NORMAL,  ANCHOR_TOP_Q)]
    cases_plateau = [("plateau", c) for c in build_cases(samples, RATIOS_PLATEAU, DAYS, NUM_CASES_PLATEAU, ANCHOR_TOP_Q)]
    cases_rebound = [("rebound", c) for c in build_cases(samples, RATIOS_REBOUND, DAYS, NUM_CASES_REBOUND, ANCHOR_TOP_Q)]

    #모든 케이스 합치기
    cases = cases_normal + cases_plateau + cases_rebound
    print(f"[cases_normal]  requested={NUM_CASES_NORMAL}, generated={len(cases_normal)}")
    print(f"[cases_plateau] requested={NUM_CASES_PLATEAU}, generated={len(cases_plateau)}")
    print(f"[cases_rebound] requested={NUM_CASES_REBOUND}, generated={len(cases_rebound)}")
    print(f"[cases_total]   total_generated={len(cases)}")

    #저장
    save_meta(cases, OUT_META)


[load_samples] loaded=2197 missing_pairs=0
[cases_normal]  requested=50, generated=50
[cases_plateau] requested=15, generated=15
[cases_rebound] requested=15, generated=15
[cases_total]   total_generated=80
[save_meta] saved -> data_wound_seg/out/meta_train_cases.csv


## 3. 데이터 구조화

In [4]:
import os
import shutil
import pandas as pd

# 주피터 노트북 기준
BASE_DIR = os.getcwd()

META_CSV = os.path.join(BASE_DIR, "data_wound_seg", "out", "meta_train_cases.csv")
OUT_ROOT = os.path.join(BASE_DIR, "data_wound_seg", "cases")
IMAGE_NAME = "image.png"
MASK_NAME = "mask.png"

def main():
    df = pd.read_csv(META_CSV)

    required_cols = {"case_id", "day", "img_path", "mask_path"}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"meta csv에 필요한 컬럼이 없음: {required_cols}")

    os.makedirs(OUT_ROOT, exist_ok=True)

    for _, row in df.iterrows():
        case_id = row["case_id"]
        day = int(row["day"])
        img_path = row["img_path"]
        mask_path = row["mask_path"]

        day_dir = os.path.join(OUT_ROOT, f"case_{case_id}", f"{day:02d}")
        os.makedirs(day_dir, exist_ok=True)

        shutil.copy(img_path, os.path.join(day_dir, IMAGE_NAME))
        shutil.copy(mask_path, os.path.join(day_dir, MASK_NAME))

    print(f"[DONE] case-wise image/mask folders created at: {OUT_ROOT}")

main()

[DONE] case-wise image/mask folders created at: /Users/baegseoyeong/OpenCV_Mask/data_wound_seg/cases
